# B7.1 — Validation-only precision policy

This short correction reuses the complete, hash-bound B7 scan caches. It preserves the failed original B7 result, selects a precision-first policy using validation layouts only, freezes it, and recomputes development confirmation. It never opens B9 and fails instead of silently repeating inference when a cache is missing or stale.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive')
PERSISTENT_ROOT = DRIVE_ROOT / 'ADVLSI2_B7'
PERSISTENT_ROOT.mkdir(parents=True, exist_ok=True)
PERSISTENT_ROOT

In [ ]:
import subprocess, sys

REPO = Path('/content/ADVLSI2_Project_updated')
BRANCH = 'agent/b7-full-layout-stitching'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, 'https://github.com/nocleo/ADVLSI2_Project_updated.git', str(REPO)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO, check=True)
    subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_b7_full_layout.py', '-v'], cwd=REPO, check=True)
print(subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip())

In [ ]:
import json, torch
print('PyTorch:', torch.__version__)

CHECKPOINT_ROOT = DRIVE_ROOT / 'ADVLSI2_B6_2/b6_multitask_unet'
required = [CHECKPOINT_ROOT / f'seed_{seed}/best.pth' for seed in (42, 43, 44)]
missing = [str(path) for path in required if not path.exists()]
assert not missing, 'Missing accepted B6.2 checkpoints: ' + ', '.join(missing)

OUTPUT_DIR = PERSISTENT_ROOT / 'b7_full_layout'
cache_files = sorted((OUTPUT_DIR / 'scan_cache').glob('*.pkl.gz'))
assert len(cache_files) == 12, f'Expected 12 complete B7 caches, found {len(cache_files)}'
current = json.loads((OUTPUT_DIR / 'summary.json').read_text())
if current.get('phase') == 'B7.1':
    original_path = OUTPUT_DIR / 'history/b7_original_failure/summary.json'
    assert original_path.is_file(), 'B7.1 exists but original B7 history is missing'
    original = json.loads(original_path.read_text())
else:
    original = current
assert original['phase'] == 'B7' and original['acceptance']['passed'] is False
assert original['untouched_b9_final_holdout_used'] is False
{'checkpoints': required, 'cache_count': len(cache_files), 'original_b7_preserved': True}

In [ ]:
command = [
    sys.executable, 'scripts/run_b7_full_layout.py',
    '--checkpoint-dir', str(CHECKPOINT_ROOT),
    '--output-dir', str(OUTPUT_DIR),
    '--device', 'cpu',
    '--phase', 'B7.1',
    '--selection-objective', 'precision_at_recall',
    '--selection-minimum-recall', '0.95',
    '--segmentation-thresholds', '0.4',
    '--reuse-scans-only',
]
print(' '.join(command))
subprocess.run(command, cwd=REPO, check=True)

In [ ]:
from IPython.display import Markdown, display

summary = json.loads((OUTPUT_DIR / 'summary.json').read_text())
assert summary['status'] == 'complete'
assert summary['official_result'] is True
assert summary['phase'] == 'B7.1'
assert summary['policy_selection']['selection_split'] == 'validation_layout_families_only'
assert summary['policy_selection']['selection_objective'] == 'precision_at_recall'
assert summary['policy_selection']['minimum_violation_recall'] == 0.95
assert summary['policy_selection']['segmentation_thresholds'] == [0.4]
assert summary['untouched_b9_final_holdout_used'] is False
display(Markdown((OUTPUT_DIR / 'README.md').read_text()))
summary['acceptance']

In [ ]:
import zipfile
from google.colab import files

archive = Path('/content/ADVLSI2_B7_1_results.zip')
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as output:
    for path in sorted(item for item in OUTPUT_DIR.rglob('*') if item.is_file()):
        relative = path.relative_to(OUTPUT_DIR)
        if relative.parts[0] in {'layout_cache', 'scan_cache'}:
            continue
        output.write(path, relative.as_posix())
print(f'Result archive: {archive.stat().st_size / 1e6:.1f} MB')
files.download(str(archive))